In [1]:
import librosa
import torch
import torch.nn.functional as F
import os
import numpy as np
from pathlib import Path
import copy # for deepcopy
import time

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel, AutoProcessor

from realtime_synth_ui import build_synth_ui # pip install "rtpysynth[ui] @ git+https://github.com/lonce/RTPySynth@v0.1.4"

In [2]:
import time

# NEW: required import for threaded version
from concurrent.futures import ThreadPoolExecutor

In [3]:
# for the rt synth
from realtime_synth.generators.base import BaseGenerator
from realtime_synth.utils import exp_map01
from realtime_synth_ui import build_synth_ui

# import the system demo synths just to have them on the interface
from realtime_synth.generators.sine import SineGenerator
from realtime_synth.generators.noisy_lp import NoisyLPGenerator

In [4]:
# for RNN4Control
from model.gru_audio_model import RNN, GRUModelConfig
from audioDataLoader.audio_dataset import  efficient_codes_to_latents, preprocess_latents_for_RNN # , latents_to_audio_simple,

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Synth/Encodec Parameters</b>

In [5]:
#audiopath = data_dir / "DSBugs--busybodyFreqFactor-00.60--c-00--x-00.wav"
# -- kbs is a training and encoding parameter only!
#Kbs = 3  # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]

g_hopsize=8   # This has a big effect on the off-line generate() because the smaller it is, the more old fashioned context switching time we spend
g_chunksize=20
sr=24000

offline_render_duration=20 #seconds
offline_render_frames=offline_render_duration*75

profiles=["syntex1", "syntex7", "nsynth2", "water"]
synthprofile=profiles[3]

if synthprofile=="syntex1" : 
    g_param_labels = ["Pitch", "Amp"]
    g_norm_param_vals = [.5, .5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:1] # the first n will be passed to the RNN generator
    
elif synthprofile=="syntex7" :
    #g_param_labels = ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "param 1", "Amp"]
    g_param_labels = ['ChirpPattern', 'DSApplause', 'DSBugs', 'DSPeepers', 'DSPistons', 'DSWind', 'TokWotalDuet', "param 1", "Amplitude"]
    g_norm_param_vals = [1,  0,     0,    0,    0,    0,    0,    0.5,      .5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:8] # the first n will be passed to the RNN generator

elif synthprofile=="nsynth2" : #nsynth
    g_param_labels = ["class", "pitch", "amp"]
    g_norm_param_vals = [0,       .5,     .5 ]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network

elif synthprofile=="water" :
    g_param_labels = ["pos"]
    g_norm_param_vals = [.5 ]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN Parameters</b>

In [6]:
# Where is the checkpoint directory?
run_directory = str(Path('./output_keep/20250926_220930_wild_WATER_CONTINUOUS'))
checkpoint_fname =   "last_checkpoint.pt" #  "checkpoint_75.pt" #"checkpoint_50.pt" # 

g_top_n = 8 #'Sample from the top N most likely outputs.'
g_temperature = .8 #'Controls the randomness of predictions.'

frame_rate=75
device='cpu'

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN CLASS</b>

In [7]:
class RNNGenerator():
    def __init__(self, checkpoint_path, model_config, data_config, enc_model, chunksize, hopsize, init_cond, top_n, temperature) : 
        #self.chkpt = chkpt

        self.clamp_val = data_config.clamp_val # need this to map between encodec latents and model input ranges

        self.model = RNN(model_config, enc_model).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

        self.enc_model=enc_model

        self.codebook_size = self.model.config.codebook_size
        self.n_q = self.model.config.n_q
        self.cond_size = self.model.config.cond_size
    

        self.chunksize=chunksize
        self.hopsize=hopsize

        self.top_n=top_n
        self.temperature=temperature

        #state between call to generate steps
        self.hidden = None # updated on every sequence step in warmup and in run_inference
        

        self.dev = next(self.enc_model.parameters()).device
        print(f'self.dev = {self.dev}, self.n_q = {self.n_q}')

        self.current_latent = self.warmup_rnn(init_cond) # initializes self.current_latent
        self.codebuf = self.run_inference(init_cond, self.chunksize) #TIME should be the last dimension


    def warmup_rnn(self, init_cond, warmup_len=10) :
        sd=.33 # to create data in [-1,1
        warmup_latents = warmup_tensor = torch.clamp(torch.randn(warmup_len, 128) * sd, -3*sd, 3*sd)  

            # Ensure warmup_latents is on the right device and has correct shape
        warmup_latents = warmup_latents.to(device)  # Shape: (warmup_len, 128)
        #warmup_len = warmup_latents.shape[0]
    
        # Handle conditioning
        if self.cond_size > 0 and init_cond is not None:
            init_cond_tensor = torch.tensor(init_cond)
            # Use first conditioning vector for entire warmup
            first_cond_vec = init_cond_tensor.unsqueeze(0).repeat(warmup_len, 1).to(device)  # (warmup_len, cond_size)
            warmup_full_input = torch.cat([warmup_latents, first_cond_vec], dim=-1)  # (warmup_len, 128 + cond_size)
        else:
            # No conditioning
            warmup_full_input = warmup_latents  # (warmup_len, 128)
        
        self.hidden = self.model.init_hidden(batch_size=1)
        # This propogates hidden state, but doesn't bring the output back down for the next input, using the warmup vectors instead.
        for i in range(len(warmup_full_input)):
            #_, self.hidden = self.model(warmup_full_input[i].unsqueeze(0), self.hidden,  batch_size=1)
            _, self.hidden, _, _ =  self.model(
            warmup_full_input[i].unsqueeze(0), 
            self.hidden,  
            batch_size=1)
        
        # Get the last latent for starting generation
        return warmup_latents[-1].unsqueeze(0)  # (1, 128)


    def run_inference(self, params, T: int):
        """
        Returns codes as a LongTensor of shape (n_q, T) on self.dev.
        Assumes:
          - self.current_latent: (1, 128) tensor on self.dev
          - self.hidden: RNN hidden state on self.dev
          - self.model forward returns (logits_list, hidden, sampled_indices, step_latent)
          - preprocess_latents_for_RNN keeps device, returns (1, 128)
        """
        n_q  = self.n_q
        dev  = self.dev

    
        # Pre-allocate output (n_q, T) long
        codes_nt = torch.empty(n_q, T, dtype=torch.long, device=dev)
    
        # Prepare conditioning tensor once (if used)
        if self.cond_size > 0 and params is not None:
            # avoid re-wrapping a tensor each step
            cond_vec = torch.as_tensor(params, device=dev).view(1, -1)  # (1, cond_size)
            has_cond = True
        else:
            cond_vec = None
            has_cond = False
    
        with torch.inference_mode():
            for t in range(T):
                # Build model input (latent || cond)
                if has_cond:
                    next_input_full = torch.cat([self.current_latent, cond_vec], dim=-1)  # (1, 128+cond)
                else:
                    next_input_full = self.current_latent                                 # (1, 128)
    
                # One forward; unified sampling inside the model
                logits_list, self.hidden, sampled_indices, step_latent = self.model(
                    next_input_full,
                    self.hidden,
                    use_teacher_forcing=False,
                    temperature=self.temperature,
                    batch_size=1,
                    sample_mode=("sample" if (self.top_n and self.top_n > 0) else "argmax"),
                    top_n=self.top_n,
                    return_step_latent=True,
                )
                # sampled_indices: (1, n_q) long
                codes_nt[:, t] = sampled_indices[0]  # write column t
    
                # advance latent for next step
                self.current_latent = preprocess_latents_for_RNN(step_latent, self.clamp_val)  # (1, 128)
    
        return codes_nt  # (n_q, T) long, on self.dev

    
    def getNextCodeChunk(self, params, hop: int):
        with torch.inference_mode():
            new_codes = self.run_inference(params, hop)  # (n_q, h) long on self.dev
    
            buf = self.codebuf           # (n_q, T)
            T   = self.chunksize
            h   = new_codes.size(1)
    
            if h >= T:
                buf.copy_(new_codes[:, -T:])        # replace with newest T
            else:
                buf[:, :-h] = buf[:, h:]            # shift left
                buf[:, -h:] = new_codes             # append tail
    
            return buf

In [8]:
# load the encoder model 
#####################################################################
enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz")
enc_model.eval()

# --- these are only for encoding, not decoding
#enc_model.config.target_bandwidths = [Kbs] # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
#processor = AutoProcessor.from_pretrained("facebook/encodec_24khz", use_fast=False)
enc_model.device

device(type='cpu')

In [9]:
# load the RNN model 
#####################################################################
config_path = os.path.join(run_directory, "config.pt")
checkpoint_path = os.path.join(run_directory, "checkpoints", checkpoint_fname) 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)
model_config = saved_configs["model_config"]
data_config = saved_configs["data_config"]

rnngen = RNNGenerator(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, g_init_cond, g_top_n, g_temperature)
print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}")
rnngen.clamp_val 

Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
self.dev = cpu, self.n_q = 8
Model successfully loaded from checkpoint.
Using device = cpu


15

In [10]:
# # Visualize hops shiting into the "right" side of chunks
# print(f"rnngen.codebuf is {rnngen.codebuf}")
# newfoo = rnngen.getNextCodeChunk(g_init_cond, g_hopsize)
# print(f"rnngen.codebuf is {rnngen.codebuf}")

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>CodecSynth</b>

In [11]:

class MyEncodecPlayer(BaseGenerator):
    # normalized params in [0,1]
    param_labels = g_param_labels

    # -----------------------
    def __init__(self, rnngen, buffersize, init_norm_params=None):

        self.units_p=np.zeros_like(init_norm_params)
        super().__init__(init_norm_params or [0.5, 0.6])  # defaults
        print(f'Initialize MyEncodecPlayer')
        self.set_params(self.norm_params)  # initialize semantic values
        

        self.rnngen = rnngen
        self.cond_size = rnngen.cond_size

        self.chunksizeframes = g_chunksize   # decode this many frames each time
        self.framehopsize    = g_hopsize     # decode a new chunk every framehopsize
        self.nextendframe    = self.framehopsize

        self.buffersize = buffersize
        self.nextsample = 0
        # NOTE: assumes global sr and frame_rate are defined elsewhere in your code
        self.framesizesamples = sr // frame_rate  # e.g., 75; encoder is 75 fps

        self.currentchunkframe = 0  # nth frame in the chunk of audio we are playing
        self.seeding_len = self.chunksizeframes - self.framehopsize
        self.genaudioframe = 0      # mth frame we've generated in total

        self._last_error = None
        self._decodetime = 0.0
        self._callrecord = ""

        # small scratch buffer to avoid per-callback allocations (optional)
        self._scratch = np.empty(self.buffersize, dtype=np.float32)

        # === Preload first hop synchronously (unchanged behavior) ===
        with torch.inference_mode():
            FOO = rnngen.getNextCodeChunk(self.norm_params[:self.cond_size], self.framehopsize)  # (n_q, T) long
        
            assert isinstance(FOO, torch.Tensor) and FOO.dtype == torch.long
            assert FOO.dim() == 2 and FOO.size(0) == rnngen.n_q
            assert FOO.device == next(enc_model.parameters()).device
        
            codes_bnt = FOO.unsqueeze(0)  # (B=1, n_q, T)
        
            # decode; some HF builds return (B,C,S), others (C,S)
            audio_t = enc_model.decode([codes_bnt], audio_scales=[None])[0]

        
            # normalize to (S,) torch tensor BEFORE converting to numpy
            if audio_t.ndim == 3:      # (B, C, S)
                audio_t = audio_t[0, 0]
            elif audio_t.ndim == 2:    # (C, S)
                audio_t = audio_t[0]
            elif audio_t.ndim == 1:    # (S,)
                pass
            else:
                raise RuntimeError(f"unexpected audio shape: {tuple(audio_t.shape)}")
        
            # final 1D numpy block; .detach() unnecessary inside inference_mode
            self.thisaudioseq = audio_t[-self.framehopsize * self.framesizesamples:] \
                                    .to("cpu", non_blocking=True) \
                                    .contiguous() \
                                    .numpy()        


        
        self.nextaudioseq = None  # will be filled by background worker

        # NEW: single background worker + a future for the next hop
        self._hop_exec = ThreadPoolExecutor(max_workers=1, thread_name_prefix="HopGen")
        self._next_future = None

        # Kick off the first async hop immediately
        self._schedule_next_hop()

    # -----------------------
    def _schedule_next_hop(self):
        """Launch getNextAudioHop() in the background (non-blocking)."""
        if self._next_future is None:
            try:
                self._next_future = self._hop_exec.submit(self.getNextAudioHop)
            except Exception as e:
                self._last_error = f"scheduling error: {e!r}"
                self._next_future = None

    # -----------------------
    def _try_collect_next(self):
        """
        If the background hop has finished, collect it into self.nextaudioseq (non-blocking).
        """
        fut = self._next_future
        if fut is not None and fut.done():
            try:
                self.nextaudioseq = fut.result()
            except Exception as e:
                self._last_error = f"hop result error: {e!r}"
                self.nextaudioseq = None
            finally:
                self._next_future = None  # allow scheduling the following hop

    # -----------------------
    def close(self):
        """Optional: call when tearing down to stop the worker quickly."""
        try:
            if hasattr(self, "_hop_exec") and self._hop_exec:
                self._hop_exec.shutdown(wait=False, cancel_futures=True)
        except Exception:
            pass

    # -----------------------
    def getNextAudioHop(self):
        """
        Heavy work: RNN inference + EnCodec decode for one hop.
        Runs on the background thread.
        Returns a 1D numpy array of length framehopsize * framesizesamples (mono, float32 or convertible).
        """
        self.genaudioframe = self.genaudioframe + self.framehopsize
        self._callrecord = self._callrecord + f";(start: {self.genaudioframe}, end: {self.genaudioframe + self.chunksizeframes})"

        start_time = time.monotonic()
        with torch.inference_mode():
            next_codes = self.rnngen.getNextCodeChunk(self.norm_params[:self.cond_size], self.framehopsize)  # (n_q, T) long, on enc_model's device
            nextseq = enc_model.decode([next_codes.unsqueeze(0)], audio_scales=[None])[0]

            
        nextseq = nextseq[0, 0].detach().numpy()
        self._decodetime += (time.monotonic() - start_time)

        # take only the hopsize of audio that we need
        return nextseq[-self.framehopsize * self.framesizesamples:]

    # -----------------------
    # This first sets the norm_params, and the units_params which are just used for display (the norm_params are the ones sent to the synth)
    def set_params(self, norm_params):
        super().set_params(norm_params)
        # Map [0,1] → semantic values (your mapping)
        # self.freq = float(exp_map01(self.norm_params[0], 20.0, 2000.0))  # exponential Hz
        # self.amp  = float(self.norm_params[1])                           # linear gain 0..1

        if synthprofile=="syntex1" : # syntex
            self.units_p[0] = self.norm_params[0]
            self.units_p[1] = self.norm_params[1]

        elif synthprofile=="syntex7" : # syntex
            self.units_p[0] = self.norm_params[0]
            self.units_p[1] = self.norm_params[1]
            self.units_p[2] = self.norm_params[2]
            self.units_p[3] = self.norm_params[3]
            self.units_p[4] = self.norm_params[4]
            self.units_p[5] = self.norm_params[5]
            self.units_p[6] = self.norm_params[6]
            self.units_p[7] = self.norm_params[7]
            self.units_p[8] = self.norm_params[8]
        
        elif synthprofile=="nsynth2" : #nsynth
            self.units_p[0] = self.norm_params[0]
            self.units_p[1] = 64 + self.norm_params[1]*12
            self.units_p[2] = self.norm_params[2]
        
        elif synthprofile=="water" : #nsynth
            self.units_p[0] = self.norm_params[0]

    
    # -----------------------
    def generate(self, frames, sr):
        assert frames == self.buffersize, "ooh, you're in trouble if frames requested is different than the buffer size."

        # if self.amp <= 0.0 or self.freq <= 0.0:
        #     self._scratch.fill(0.0)
        #     return self._scratch

        # slice current hop
        endsamp = self.nextsample + self.buffersize
        y = self.thisaudioseq[self.nextsample:endsamp]
        self.nextsample = endsamp

        # NON-BLOCKING: if we just started a hop, see if the background result is ready
        if self.currentchunkframe == 0:
            self._try_collect_next()
            # if nothing in-flight, (re)start background worker
            if self._next_future is None:
                self._schedule_next_hop()

        # advance within hop; at hop boundary, try to swap
        self.currentchunkframe += 1
        if self.currentchunkframe == self.framehopsize:
            if self.nextaudioseq is not None:
                # swap in new hop (no copy; ensure float32)
                self.thisaudioseq = np.asarray(self.nextaudioseq, dtype=np.float32)
                self.nextaudioseq = None
                self._schedule_next_hop()  # immediately start computing the following hop
            else:
                # no hop ready → output SILENCE for one hop (your preference)
                msg = "missed hop swap"
                self._last_error = (self._last_error + " | " + msg) if self._last_error else msg
                self.thisaudioseq = np.zeros(self.framehopsize * self.framesizesamples, dtype=np.float32)
                # also (re)schedule next hop in case worker died
                if self._next_future is None:
                    self._schedule_next_hop()
            # reset for new hop window
            self.currentchunkframe = 0
            self.nextsample = 0

        # # Do post signal processing based on params if you need to 
        # # scale into scratch (avoids alloc every block)
        # np.multiply(y, self.amp, out=self._scratch, casting='unsafe')
        # return self._scratch

        #otherwise, just return the buffer
        return y

    # -----------------------
    def formatted_readouts(self):
        # Optional: pretty labels shown next to sliders
        if synthprofile=="syntex1": # syntex
            return [
                # f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                # f"{self.param_labels[1]}: {self.amp:.3f}",
                f"{self.param_labels[0]}: {self.units_p[0]:1.1f} ",
                f"{self.param_labels[1]}: {self.units_p[1]:1.1f}"
            ]

        if synthprofile=="syntex7": # syntex
            return [
                # f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                # f"{self.param_labels[1]}: {self.amp:.3f}",
                f"{self.param_labels[0]}: {self.units_p[0]:1.1f} ",
                f"{self.param_labels[1]}: {self.units_p[1]:1.1f}"
                f"{self.param_labels[2]}: {self.units_p[2]:1.1f}"
                f"{self.param_labels[3]}: {self.units_p[3]:1.1f}"
                f"{self.param_labels[4]}: {self.units_p[4]:1.1f}"
                f"{self.param_labels[5]}: {self.units_p[5]:1.1f}"
                f"{self.param_labels[6]}: {self.units_p[6]:1.1f}"
                f"{self.param_labels[7]}: {self.units_p[7]:1.1f}"
                f"{self.param_labels[8]}: {self.units_p[8]:1.1f}"
            ]

        
        elif synthprofile=="nsynth2" : # nsynth
            return [
                # f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                # f"{self.param_labels[1]}: {self.amp:.3f}",
                f"{self.param_labels[0]}: {self.units_p[0]:1.1f} Hz",
                f"{self.param_labels[1]}: {self.units_p[1]:1.1f}",
                f"{self.param_labels[0]}: {self.units_p[2]:1.1f} Hz"               
                ]
                
        elif synthprofile=="water" : #nsynth
            return [
                f"{self.param_labels[0]}: {self.units_p[0]:1.1f} level"
            ]


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>ON LINE, Realtime TEST</b>

In [12]:
GENS = {
    "MyEncodecPlayer": lambda: MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals),
    "Sine": SineGenerator,            # defined in the default system
    "Noisy LP": NoisyLPGenerator,     # defined in the default system
}
print(f"Will use samplerate = {sr} and blocksize = {buffersize}")

print(f"g_norm_param_vals = {g_norm_param_vals}")
synth, ui = build_synth_ui(GENS, samplerate=sr, blocksize=buffersize, channels=1)

Will use samplerate = 24000 and blocksize = 320
g_norm_param_vals = [0.5]
Initialize MyEncodecPlayer


HTML(value='')

In [13]:
print(getattr(synth.gen, "_last_error", None))

None


In [14]:
print(getattr(synth.gen, "_callrecord ", None))

None


In [15]:
print(getattr(synth.gen, "__decodetime", None))

None


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST</b>

In [16]:
# just checking, not passed to to the synth which takes classes
foo=MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals)
foo.thisaudioseq
len(foo.thisaudioseq)
#display(Audio(foo.thisaudioseq, rate=sr))

Initialize MyEncodecPlayer


2560

In [17]:
# chunk=foo.thisaudioseq
# for i in range(0,1) :
#     foo.getNextAudioHop()
#     chunk = np.concatenate((chunk, foo.thisaudioseq), axis=0) 
# len(chunk)

In [18]:
c=[]

hops=int(offline_render_duration*frame_rate/g_hopsize)

for hopnum in range(0,hops) :
    c=  np.concatenate((c , foo.thisaudioseq), axis=0) 
    foo.thisaudioseq=foo.getNextAudioHop()
print(f"time spent decoding {offline_render_duration} secs of audio = {foo._decodetime:.2f}")


time spent decoding 20 secs of audio = 3.47


In [19]:
display(Audio(c, rate=sr))

In [20]:
#print(f"{foo._callrecord}")

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST 2 - calling generate</b>

In [21]:
bar= MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals)

Initialize MyEncodecPlayer


In [22]:
#you can't just call generate in a loop like this because the calls happen way to fast for the NN to meet the needs of the fill buffer callback
# for buffernum in range(0,1375) :
#     audio_out =  np.concatenate((audio_out, bar.generate(buffersize, sr)), axis=0) 
    

In [23]:
#This function uses generate (but waits for the "next hop" generation from the neural net so it doesn't lose any audio.
# You can think of this as rendering as fast as it can - faster than real time, but only as fast as the NN can produce the data
#
def render_offline(player, n_blocks, sr, buffersize):
    blocks = []
    for i in range(n_blocks):
        # if we’re at the hop boundary, ensure next hop is ready
        if player.currentchunkframe == 0:
            # make sure a future is scheduled
            if player._next_future is None:
                player._schedule_next_hop()
            # wait here until it finishes, then collect
            if player._next_future is not None:
                player._next_future.result()   # blocks here offline
                player._try_collect_next()
        blocks.append(player.generate(buffersize, sr).copy())
    return np.concatenate(blocks).astype(np.float32)

# usage:
#audio_out = render_offline(bar, 1375, sr, buffersize)

In [24]:

start_time = time.monotonic()
audio_out = render_offline(bar, offline_render_frames, sr, buffersize)
rnnelapsed_time = time.monotonic() - start_time
print(f"RNN time to generate {offline_render_duration} secs of sound: {rnnelapsed_time:.2f}. Converting to audio...")


RNN time to generate 20 secs of sound: 2.97. Converting to audio...


In [25]:
display(Audio(audio_out, rate=sr)) 